# conformance_engine\nGenerated from 02_processing/conformance_engine.py for Databricks notebook import.\n

In [ ]:
"""Production-oriented conformance engine driven by mapping metadata."""\n\nfrom __future__ import annotations\n\nimport logging\nfrom dataclasses import dataclass\nfrom typing import Any, Mapping\n\nfrom pyspark.sql import DataFrame\nfrom pyspark.sql import functions as F\n\nlogger = logging.getLogger(__name__)\n\n_FRAMEWORK_COLS = frozenset(\n    {\n        "source_system",\n        "source_entity",\n        "load_id",\n        "ingest_ts",\n        "bronze_dq_status",\n        "bronze_dq_failed_check",\n        "dq_status",\n        "dq_failed_rule",\n    }\n)\n\n_REQUIRED_MAPPING_KEYS = {"transform_expression", "conformance_column"}\n_ALLOWED_WRITE_MODES = {"append", "overwrite", "error", "ignore"}\n_ALLOWED_TABLE_TYPES = {"managed", "external"}\n\n\nclass ConformanceConfigError(ValueError):\n    """Raised when conformance configuration is invalid."""\n\n\nclass ConformanceExecutionError(RuntimeError):\n    """Raised when conformance execution fails."""\n\n\ndef _require_non_empty(value: Any, field_name: str) -> str:\n    if value is None:\n        raise ConformanceConfigError(f"{field_name} is required")\n    text = str(value).strip()\n    if not text:\n        raise ConformanceConfigError(f"{field_name} is required")\n    return text\n\n\ndef _normalize_bool(value: Any) -> str:\n    if isinstance(value, bool):\n        return str(value).lower()\n    if isinstance(value, str):\n        text = value.strip().lower()\n        if text in {"true", "false"}:\n            return text\n    raise ConformanceConfigError(f"Expected boolean-compatible value, got: {value!r}")\n\n\n@dataclass(frozen=True)\nclass MappingRule:\n    transform_expression: str\n    conformance_column: str\n    ordinal: int\n    source_column: str | None = None\n    is_active: bool = True\n\n    @classmethod\n    def from_row(cls, row: Mapping[str, Any], ordinal: int) -> "MappingRule | None":\n        is_active_raw = row.get("is_active", True)\n\n        if isinstance(is_active_raw, str):\n            is_active = is_active_raw.strip().lower() not in {"false", "0", "n", "no"}\n        else:\n            is_active = bool(is_active_raw)\n\n        if not is_active:\n            return None\n\n        transform_expression = row.get("transform_expression")\n        conformance_column = row.get("conformance_column")\n\n        if transform_expression is None and conformance_column is None:\n            return None\n\n        if not transform_expression or not str(transform_expression).strip():\n            raise ConformanceConfigError(\n                f"mapping_rows[{ordinal}].transform_expression is required for active mappings"\n            )\n\n        if not conformance_column or not str(conformance_column).strip():\n            raise ConformanceConfigError(\n                f"mapping_rows[{ordinal}].conformance_column is required for active mappings"\n            )\n\n        return cls(\n            transform_expression=str(transform_expression).strip(),\n            conformance_column=str(conformance_column).strip(),\n            ordinal=ordinal,\n            source_column=str(row.get("source_column")).strip() if row.get("source_column") else None,\n            is_active=True,\n        )\n\n\ndef validate_mapping_rows(mapping_rows: list[dict[str, Any]]) -> list[str]:\n    errors: list[str] = []\n    seen_output_columns: set[str] = set()\n\n    for idx, row in enumerate(mapping_rows):\n        if not isinstance(row, dict):\n            errors.append(f"mapping_rows[{idx}] must be an object")\n            continue\n\n        is_active_raw = row.get("is_active", True)\n        if isinstance(is_active_raw, str):\n            is_active = is_active_raw.strip().lower() not in {"false", "0", "n", "no"}\n        else:\n            is_active = bool(is_active_raw)\n\n        if not is_active:\n            continue\n\n        missing = [key for key in _REQUIRED_MAPPING_KEYS if not row.get(key)]\n        if missing:\n            errors.append(f"mapping_rows[{idx}] missing required fields: {missing}")\n            continue\n\n        output_col = str(row["conformance_column"]).strip()\n        if output_col in seen_output_columns:\n            errors.append(f"Duplicate conformance_column found: {output_col}")\n        else:\n            seen_output_columns.add(output_col)\n\n    return errors\n\n\ndef _build_mapping_rules(mapping_rows: list[dict[str, Any]]) -> list[MappingRule]:\n    rules: list[MappingRule] = []\n\n    for idx, row in enumerate(mapping_rows):\n        if not isinstance(row, dict):\n            raise ConformanceConfigError(f"mapping_rows[{idx}] must be an object")\n\n        rule = MappingRule.from_row(row, ordinal=idx)\n        if rule is not None:\n            rules.append(rule)\n\n    output_cols = [rule.conformance_column for rule in rules]\n    duplicates = sorted({col for col in output_cols if output_cols.count(col) > 1})\n    if duplicates:\n        raise ConformanceConfigError(\n            f"Duplicate conformance_column values are not allowed: {duplicates}"\n        )\n\n    return rules\n\n\ndef apply_column_mappings(\n    df: DataFrame,\n    mapping_rows: list[dict[str, Any]],\n    passthrough_on_empty: bool = True,\n    framework_cols: set[str] | frozenset[str] | None = None,\n) -> DataFrame:\n    framework_cols = framework_cols or _FRAMEWORK_COLS\n\n    validation_errors = validate_mapping_rows(mapping_rows)\n    if validation_errors:\n        raise ConformanceConfigError(\n            f"Invalid mapping metadata: {validation_errors}"\n        )\n\n    rules = _build_mapping_rules(mapping_rows)\n\n    if not rules:\n        if not passthrough_on_empty:\n            raise ConformanceConfigError(\n                "No active valid mapping rows found and passthrough_on_empty=False"\n            )\n\n        passthrough_cols = [c for c in df.columns if c not in framework_cols]\n\n        logger.warning(\n            "No active valid mapping rows found. Returning passthrough DataFrame with framework columns removed."\n        )\n\n        return df.select(*passthrough_cols) if passthrough_cols else df\n\n    try:\n        expressions = [\n            F.expr(rule.transform_expression).alias(rule.conformance_column)\n            for rule in rules\n        ]\n        result_df = df.select(*expressions)\n\n        logger.info(\n            "Applied %s conformance mapping rules",\n            len(rules),\n        )\n\n        return result_df\n\n    except Exception as exc:\n        raise ConformanceExecutionError(\n            "Conformance select failed. One or more transform_expression values are invalid. "\n            f"Original error: {exc}"\n        ) from exc\n\n\ndef write_conformance(\n    df: DataFrame,\n    table_name: str,\n    table_type: str = "managed",\n    external_path: str | None = None,\n    mode: str = "append",\n    merge_schema: bool = True,\n) -> None:\n    resolved_table_name = _require_non_empty(table_name, "table_name")\n    resolved_table_type = _require_non_empty(table_type, "table_type").lower()\n    resolved_mode = _require_non_empty(mode, "mode").lower()\n\n    if resolved_table_type not in _ALLOWED_TABLE_TYPES:\n        raise ConformanceConfigError(\n            f"Unsupported table_type '{resolved_table_type}'. Supported values: {sorted(_ALLOWED_TABLE_TYPES)}"\n        )\n    resolved_external_path = (\n        _require_non_empty(external_path, "external_path") if external_path else None\n    )\n    if resolved_table_type == "external" and not resolved_external_path:\n        raise ConformanceConfigError("external_path is required when table_type=external")\n\n    if resolved_mode not in _ALLOWED_WRITE_MODES:\n        raise ConformanceConfigError(\n            f"Unsupported write mode '{resolved_mode}'. Supported modes: {sorted(_ALLOWED_WRITE_MODES)}"\n        )\n\n    try:\n        writer = (\n            df.write.mode(resolved_mode)\n            .format("delta")\n            .option("mergeSchema", _normalize_bool(merge_schema))\n        )\n        if resolved_table_type == "external" and resolved_external_path:\n            writer = writer.option("path", resolved_external_path)\n        writer.saveAsTable(resolved_table_name)\n\n        logger.info(\n            "Wrote conformance DataFrame to table=%s mode=%s mergeSchema=%s",\n            resolved_table_name,\n            resolved_mode,\n            merge_schema,\n        )\n\n    except Exception as exc:\n        raise ConformanceExecutionError(\n            f"Failed to write conformance DataFrame to table={resolved_table_name}: {exc}"\n        ) from exc\n